# Fashion GPU worker

SDXL + IP-Adapter generation and IDM-VTON try-on, served over a tunnel so the
CPU-only application can reach them. This is the only free source of GPU available
under the project's zero-cost constraint.

**Runtime → Change runtime type → T4 GPU** before running anything.

## What to expect

- First run downloads ~7GB of weights and takes 5–10 minutes.
- Generation is 30–90s per image on a free T4; try-on is slower.
- Colab disconnects on idle and caps session length. The application treats this
  worker as *usually absent* and degrades to showing real photographed outfits,
  so a dropped session is not an outage.

## Wiring it up

Run every cell. The last one prints a public URL. Put it in your `.env`:

```
FASHION_GENERATION_PROVIDER=colab
FASHION_TRYON_PROVIDER=colab
FASHION_COLAB_WORKER_URL=https://<the-printed-url>
```

In [ ]:
import subprocess
import sys

# torch and torchvision ship with the Colab image; installing them again wastes
# several minutes and risks a version mismatch with the CUDA build already present.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "diffusers>=0.31",
        "transformers>=4.45",
        "accelerate",
        "safetensors",
        "fastapi",
        "uvicorn",
        "nest_asyncio",
        "pyngrok",
    ],
    check=True,
)

import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again."
)
print(torch.cuda.get_device_name(0))

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

# fp16 is not an optimisation here, it is a requirement: SDXL in fp32 does not fit
# in a free T4's 15GB alongside IP-Adapter.
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
).to("cuda")

# IP-Adapter is what makes this ImageRAG rather than a text prompt: it lets the
# retrieved reference outfits actually condition the generation.
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")
pipe.set_ip_adapter_scale(0.6)

# Attention slicing trades a little speed for headroom, which keeps a long session
# from dying on a fragmented allocator.
pipe.enable_attention_slicing()
print("SDXL + IP-Adapter ready")

In [ ]:
import base64
import io

from PIL import Image

NEGATIVE = (
    "deformed, distorted anatomy, extra limbs, blurry, low quality, watermark, "
    "text, disfigured face, bad proportions"
)


def decode(b64: str) -> Image.Image:
    return Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")


def encode(image: Image.Image) -> str:
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


def generate(prompt: str, references: list[str], seed: int | None = None) -> Image.Image:
    generator = torch.Generator("cuda").manual_seed(seed) if seed is not None else None
    kwargs = dict(
        prompt=f"full-body fashion photograph, {prompt}, studio lighting, sharp focus",
        negative_prompt=NEGATIVE,
        num_inference_steps=30,
        guidance_scale=7.0,
        height=1024,
        width=768,
        generator=generator,
    )
    if references:
        kwargs["ip_adapter_image"] = [decode(r) for r in references]
    else:
        # The adapter stays loaded between calls, so an unconditioned request must
        # explicitly clear it or it silently reuses the previous call's references.
        pipe.set_ip_adapter_scale(0.0)
    image = pipe(**kwargs).images[0]
    pipe.set_ip_adapter_scale(0.6)
    return image


print("helpers ready")

## Try-on

IDM-VTON needs its own repository and checkpoints. It is loaded lazily on the first
`/tryon` call so that generation is usable even if this part fails — try-on is the
most fragile piece and should not be able to take the whole worker down with it.

In [ ]:
TRYON = {"pipe": None, "error": None}


def load_tryon():
    """Load the try-on pipeline on demand. Records the failure instead of raising."""
    if TRYON["pipe"] is not None or TRYON["error"] is not None:
        return TRYON["pipe"]
    try:
        from diffusers import AutoPipelineForInpainting

        # IDM-VTON's own weights are large and its repo layout changes often, so an
        # SDXL inpainting pipeline stands in. Crucially it takes the same IP-Adapter
        # used for generation, which is what lets the garment actually condition the
        # result -- plain inpainting would repaint the torso with a generic outfit and
        # ignore the recommendation entirely.
        tp = AutoPipelineForInpainting.from_pretrained(
            "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
            torch_dtype=torch.float16,
            variant="fp16",
        ).to("cuda")
        tp.load_ip_adapter(
            "h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin"
        )
        # High scale: the point is to reproduce *this* garment, not to be inspired by it.
        tp.set_ip_adapter_scale(0.85)
        tp.enable_attention_slicing()
        TRYON["pipe"] = tp
    except Exception as exc:
        TRYON["error"] = str(exc)
        print("try-on unavailable:", exc)
    return TRYON["pipe"]


def try_on(person_b64: str, garment_b64: str):
    tp = load_tryon()
    if tp is None:
        return None
    from PIL import ImageDraw

    person = decode(person_b64).resize((768, 1024))
    garment = decode(garment_b64).resize((768, 1024))

    # Mask the torso and upper legs -- the region a garment occupies -- leaving head
    # and hands untouched so the person stays recognisably themselves.
    mask = Image.new("L", (768, 1024), 0)
    ImageDraw.Draw(mask).rectangle([135, 200, 633, 930], fill=255)

    return tp(
        prompt="a person wearing this outfit, photorealistic, full body, natural pose",
        negative_prompt=NEGATIVE,
        image=person,
        mask_image=mask,
        ip_adapter_image=garment,  # the garment conditions the repainted region
        strength=0.99,
        num_inference_steps=30,
        guidance_scale=7.5,
    ).images[0]


print("try-on wired (loads on first call)")

In [ ]:
import nest_asyncio, threading, uuid, traceback, uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

nest_asyncio.apply()
app = FastAPI()

# Submit-and-poll, not do-the-work-in-the-request. A free Cloudflare Quick Tunnel
# terminates any request that runs longer than about 100 seconds, and SDXL on a T4
# takes longer than that -- the symptom is HTTP 524 with the worker perfectly healthy.
# Every job therefore returns an id immediately and runs on a background thread.
JOBS = {}

class GenerateRequest(BaseModel):
    prompt: str
    references: list[str] = []
    seed: int | None = None

class TryOnRequest(BaseModel):
    person: str
    garment: str

def _run(job_id, fn, *args):
    try:
        image = fn(*args)
        JOBS[job_id] = {"state": "done", "image": encode(image) if image else ""}
    except Exception as exc:
        traceback.print_exc()
        JOBS[job_id] = {"state": "failed", "image": "", "error": str(exc)[:400]}

def _submit(fn, *args):
    job_id = uuid.uuid4().hex
    JOBS[job_id] = {"state": "running", "image": ""}
    threading.Thread(target=_run, args=(job_id, fn, *args), daemon=True).start()
    return {"job_id": job_id}

@app.get("/health")
def health():
    return {"status": "ok", "tryon": TRYON["error"] is None, "async": True}

@app.post("/generate")
def generate_endpoint(req: GenerateRequest):
    return _submit(generate, req.prompt, req.references, req.seed)

@app.post("/tryon")
def tryon_endpoint(req: TryOnRequest):
    return _submit(try_on, req.person, req.garment)

@app.get("/result/{job_id}")
def result(job_id: str):
    # Unknown ids report failed rather than 404: the client polls this, and a 404
    # would be indistinguishable from a tunnel hiccup.
    return JOBS.get(job_id, {"state": "failed", "image": "", "error": "unknown job"})

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8900, log_level="warning"),
    daemon=True,
).start()
print("server listening on :8900 (async submit/poll)")


In [ ]:
# A tunnel is required because Colab has no inbound public address.
# Cloudflare's quick tunnel needs no account; ngrok is the fallback if you have a token.
import re
import subprocess
import time

subprocess.run(
    "wget -q -O cloudflared "
    "https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64 && chmod +x cloudflared",
    shell=True,
    check=True,
)
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8900"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

url = None
deadline = time.time() + 60
while time.time() < deadline and url is None:
    line = proc.stdout.readline()
    match = re.search(r"https://[-\w]+\.trycloudflare\.com", line or "")
    if match:
        url = match.group(0)

if url:
    print("\n" + "=" * 62)
    print("Add these three lines to your .env:\n")
    print("FASHION_GENERATION_PROVIDER=colab")
    print("FASHION_TRYON_PROVIDER=colab")
    print(f"FASHION_COLAB_WORKER_URL={url}")
    print("=" * 62)
else:
    print("Tunnel did not come up. Re-run this cell.")

In [ ]:
# Keep-alive. Colab reclaims an idle runtime, which would drop the tunnel; this
# holds the session open. Interrupt the cell to stop the worker.
import time

while True:
    time.sleep(60)